In [2]:
import os
import sys
import joblib

sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from torch.utils.data import DataLoader, random_split, TensorDataset

from utils.models import CNNLSTM
from utils.preprocessing import (
    balance_dataset,
    FEATURES
)
from utils.evaluation import evaluate_model
from property_factory import build_properties
import property_driven_ml.logics as pml_logics

In [3]:
# =========================
# Define datasets
# =========================
# df_cicids2017 = df = pd.read_csv("../data/cicids2017_preprocessed.tsv", on_bad_lines="skip", delimiter="\t")
df_ciciot2023 = df = pd.read_csv("../data/ciciot2023_preprocessed.tsv", on_bad_lines="skip", delimiter="\t")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [11]:
df_ddos_syn = df_ciciot2023[df_ciciot2023["label"] == "DDOS_SYN_Flood"]
df_other = df_ciciot2023[df_ciciot2023["label"] != "DDOS_SYN_Flood"]

# Keep only 10% of DDoS-SYN_Flood
df_ddos_sampled = df_ddos_syn.sample(frac=0.1, random_state=42)

# Combine back
df_ciciot2023 = pd.concat([df_other, df_ddos_sampled], ignore_index=True)

In [5]:
DATASETS = {
    # "cicids2017": {"data": df_cicids2017},
    "ciciot2023": {"data": df_ciciot2023},
}

In [6]:
BATCH_SIZE = 256
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
TEST_SIZE = 0.3
LAMBDA_PROP = 0.3
WINDOW_SECONDS = 5.0

In [12]:
for dataset_name, dataset_config in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"TRAINING ON: {dataset_name.upper()}")
    print("=" * 70)

    df = dataset_config["data"].copy()

    feature_cols = [c for c in FEATURES if c in df.columns]

    print("Shape:", df.shape)
    print(df["label"].value_counts())
    print("Features:", len(feature_cols))

    X = df[feature_cols].copy()
    y = df["label"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=42,
    )

    X_train_balanced, y_train_balanced = balance_dataset(X_train, y_train)
    print("Balanced training set shape:", X_train_balanced.shape)
    print("Balanced training label distribution:\n", pd.Series(y_train_balanced).value_counts())

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train_balanced)
    y_test_enc = label_encoder.transform(y_test)

    X_train_bal = X_train_balanced.copy()
    X_test_local = X_test.copy()

    categorical_cols = X_train_bal.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
    ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    if len(categorical_cols) > 0:
        X_train_bal[categorical_cols] = ordinal_encoder.fit_transform(X_train_bal[categorical_cols])
        X_test_local[categorical_cols] = ordinal_encoder.transform(X_test_local[categorical_cols])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal[feature_cols])
    X_test_scaled = scaler.transform(X_test_local[feature_cols])

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).unsqueeze(1)

    y_train_tensor = torch.tensor(y_train_enc, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test_enc, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    num_classes = len(label_encoder.classes_)
    model = CNNLSTM(n_features=len(feature_cols), num_classes=num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    properties = build_properties(
        device=device,
        scaler=scaler,
        feature_names=feature_cols,
        label_encoder=label_encoder,
        logic=pml_logics.GoedelFuzzyLogic()
    )

    x_batch, _ = next(iter(train_loader))
    x_batch = x_batch.to(device)
    prop_loss, prop_stats = properties.compute_loss(model, x_batch)

    print("Initial property loss:", prop_loss.item())
    print("Property stats:", prop_stats)

    history = []

    for epoch in range(NUM_EPOCHS):
        model.train()

        pred_losses = []
        prop_losses = []
        last_stats = {}

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(x_batch)
            pred_loss = criterion(logits, y_batch)

            prop_loss, prop_stats = properties.compute_loss(model, x_batch)
            
            # For the first few epochs, focus on prediction loss to allow the model to learn basic patterns. 
            # After that, start incorporating the property loss to guide the model towards satisfying the properties.
            if epoch < 3:
                loss = pred_loss
            else:
                loss = pred_loss + LAMBDA_PROP * prop_loss
            loss.backward()
            optimizer.step()

            pred_losses.append(pred_loss.item())
            prop_losses.append(prop_loss.item())
            last_stats = prop_stats

        avg_pred_loss = float(np.mean(pred_losses))
        avg_prop_loss = float(np.mean(prop_losses))

        history.append({
            "epoch": epoch + 1,
            "pred_loss": avg_pred_loss,
            "prop_loss": avg_prop_loss,
            **last_stats,
        })

        print(
            f"Epoch {epoch+1:02d} | "
            f"pred_loss={avg_pred_loss:.4f} | "
            f"prop_loss={avg_prop_loss:.4f} | "
            f"{', '.join(f'{k}={v:.4f}' for k, v in last_stats.items())}"
        )

    # evaluation
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)

            logits = model(x_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(y_batch.cpu().numpy())

    y_true_labels = label_encoder.inverse_transform(np.array(all_true))
    y_pred_labels = label_encoder.inverse_transform(np.array(all_preds))

    evaluate_model(
        y_true_labels,
        y_pred_labels,
        model_name=f"{dataset_name}-trained CNN-LSTM with properties",
    )

    os.makedirs("models", exist_ok=True)
    save_path = f"models/cnnlstm_property_{dataset_name}.joblib"

    joblib.dump(
        {
            "model": model.cpu(),
            "ordinal_encoder": ordinal_encoder,
            "scaler": scaler,
            "label_encoder": label_encoder,
            "features": feature_cols,
            "categorical_cols": list(categorical_cols),
        },
        save_path,
    )

    print(f"\nSaved model to: {save_path}")

    history_df = pd.DataFrame(history)
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["pred_loss"], label="Prediction loss")
    plt.plot(history_df["epoch"], history_df["prop_loss"], label="Property loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training losses - {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


TRAINING ON: CICIOT2023
Shape: (14518511, 47)
label
DDOS_SYN_FLOOD    7412700
ATTACK            4748328
DOS_HTTP_FLOOD    1508589
BENIGN             342255
DDOS_UDP_FLOOD     290106
PORTSCAN           216533
Name: count, dtype: int64
Features: 33


KeyboardInterrupt: 